# Inflation guard for our own rule (no learned part)
Problem found in `Ours_vs_T10_*`: if every candidate gets more inliers (x1.25 to x2), our bars (8 / 11) and the agreement inlier floor (3) answer look-alikes (fresh NUDT: 2 wrong at x1.5, 5 at x2).
Guard: measure, label-free, how many inliers a WRONG place gets on this run (mean inliers of the candidates more than 4 sigma from the choice, running mean over the photos so far, started at a dev prior, as in T10 `scale_robust`), and scale the three inlier numbers by max(1, level / prior).
Prior = the larger of the far levels of our official150 and nudt707 sets (so clean data is not scaled); test set = fresh nudt1010 (not used to set the prior). Nothing is fitted to labels.

In [ ]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 220)
from train.checks import replay
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
T7 = ROOT / "tests/Test_Iteration_7/results"
SETS = {"official150": "official150", "nudt707": "nudt_fresh707", "nudt1010": "nudt_fresh1010"}
photos = {s: replay.load_set(T7 / "rich_dump" / f) for s, f in SETS.items()}
raw = pd.concat([pd.read_csv(T7 / f"signals_ours_{s}.csv.gz") for s in SETS], ignore_index=True)
SCALES = [1.0, 1.25, 1.5, 2.0]
WEIGHT = 30
levels = sr.far_levels(raw)
prior = max(levels[levels.data == s].far_inl.mean() for s in ["official150", "nudt707"])
print({s: round(levels[levels.data == s].far_inl.mean(), 3) for s in SETS}, "prior", round(prior, 3))

In [ ]:
def running_scale(f):
    """per (data, photo): factor max(1, running far level / prior) when every inlier count is multiplied by f."""
    lv = levels.assign(far_inl=levels.far_inl * f)
    out = {}
    for data, g in lv.sort_values(["data", "photo"]).groupby("data", sort=False):
        k = np.arange(1, len(g) + 1)
        run = (WEIGHT * prior + g.far_inl.cumsum().to_numpy()) / (WEIGHT + k)
        for photo, r in zip(g.photo, run):
            out[(data, photo)] = max(1.0, r / prior)
    return out

def ours(f, guard):
    scale = running_scale(f) if guard else {}
    rows = []
    for data, ph in photos.items():
        for pid, photo in ph.items():
            s = scale.get((data, pid), 1.0)
            inflated = [np.round(np.asarray(q["inliers"], float) * f) for q in photo["queries"]]
            r = replay.decide_current(photo, inliers=inflated, bar=round(8 * s), escalation_bar=round(11 * s), agree_query_inliers=round(3 * s))
            rows.append(dict(data=data, photo=pid, stage=r["stage"], pts=int(r["points"]), err_sigma=r["err_sigma"], n_queries=r["n_queries"]))
    return pd.DataFrame(rows)

rows = []; KEEP = {}
for f in SCALES:
    for guard in [False, True]:
        d = ours(f, guard); KEEP[(f, guard)] = d
        s = cj.summary(d)
        for data in s.index:
            rows.append(dict(scale=f, guard=guard, data=data, score=s.loc[data, "score"], wrong=int(s.loc[data, "wrong"]), answered=s.loc[data, "answered"], queries=s.loc[data, "queries"]))
TAB = pd.DataFrame(rows)
for data in SETS:
    d = TAB[TAB.data == data]
    print(data)
    display(pd.concat({"score": d.pivot_table(index="guard", columns="scale", values="score").round(2), "wrong": d.pivot_table(index="guard", columns="scale", values="wrong").astype(int),
                       "answered": d.pivot_table(index="guard", columns="scale", values="answered").round(3)}, axis=1))

In [ ]:
gain = []
for f in SCALES:
    g = cj.paired_gain(KEEP[(f, False)], KEEP[(f, True)], list(SETS))
    gain.append(dict(scale=f, **g))
TAB.to_csv(T7 / "ours_inflation_guard.csv", index=False)
pd.DataFrame(gain)